# Fine-tuning 전후 비교 평가

`2. QLoRA Fine-tuning.ipynb`에서 학습한 어댑터를 base 모델과 **정량적으로 비교**합니다. fine-tuning 전후를 같은 평가셋·같은 프롬프트로 비교하는 과정을 실습합니다.


In [1]:
import os

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import torch
from dotenv import load_dotenv
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration
from peft import PeftModel
from qwen_vl_utils import process_vision_info

import vlm_finetune_utils as vu

load_dotenv()
HF_TOKEN = os.environ["HF_TOKEN"]
HF_DATASET_REPO_ID = os.environ["HF_DATASET_REPO_ID"]

MODEL_ID = "Qwen/Qwen3-VL-2B-Instruct"
ADAPTER_DIR = "outputs/qlora-chartqa-adapter"

# 학습 때와 동일한 상한 — 이미지 토큰 수를 억제해 두 모델을 동시에 올려도 메모리를 아낍니다.
MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = 512 * 28 * 28

N_EVAL = 50  # 강의 시간 내 실습을 위한 서브셋 크기 (최대 400)
vu.download_dataset_from_hub("data", HF_DATASET_REPO_ID, token=HF_TOKEN)
val_records = vu.load_jsonl("data/val/annotations.jsonl")[:N_EVAL]


D:\develop\github\course_VLM\5. VLM Fine Tuning\examples\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. base 모델과 (base + QLoRA 어댑터) 모델 로드

In [2]:
processor = AutoProcessor.from_pretrained(
    MODEL_ID, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS
)

base_model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto"
)

tuned_model = PeftModel.from_pretrained(
    Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto"
    ),
    ADAPTER_DIR,
)


D:\develop\github\course_VLM\5. VLM Fine Tuning\examples\.venv\Lib\site-packages\accelerate\utils\modeling.py:817: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  _ = torch.tensor([0], device=i)


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/625 [00:00<01:30,  6.92it/s]

Loading weights:   8%|▊         | 47/625 [00:00<00:02, 227.40it/s]

Loading weights:  15%|█▌        | 95/625 [00:00<00:01, 330.87it/s]

Loading weights:  23%|██▎       | 146/625 [00:00<00:01, 393.82it/s]

... (13 outputs left out)

Loading weights:  46%|████▌     | 289/625 [00:00<00:00, 451.70it/s]

Loading weights:  67%|██████▋   | 418/625 [00:00<00:00, 714.81it/s]

Loading weights:  96%|█████████▌| 600/625 [00:00<00:00, 1039.41it/s]

Loading weights: 100%|██████████| 625/625 [00:00<00:00, 642.69it/s] 

## 2. 추론 함수

In [3]:
def ask(model, record, image_root):
    messages = vu.to_chat_messages(record, image_root)[:1]  # human 턴만 전달
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, _ = process_vision_info(messages)
    inputs = processor(
        text=[text], images=image_inputs, return_tensors="pt", padding=True
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=64)
    trimmed = output_ids[:, inputs["input_ids"].shape[1]:]
    return processor.batch_decode(trimmed, skip_special_tokens=True)[0].strip()


## 3. 검증셋 전체에 대해 두 모델 비교

In [4]:
rows = []
for r in val_records:
    base_answer = ask(base_model, r, "data/val")
    tuned_answer = ask(tuned_model, r, "data/val")
    rows.append(
        {
            "question": r["question"],
            "gold": r["answer"],
            "base_answer": base_answer,
            "base_correct": vu.judge_answer(base_answer, r["answer"]),
            "tuned_answer": tuned_answer,
            "tuned_correct": vu.judge_answer(tuned_answer, r["answer"]),
        }
    )

for row in rows[:5]:
    print(row)


{'question': '연한 파란색은 무엇을 나타내나요?', 'gold': '페이스북 팬', 'base_answer': '페이스북 팬', 'base_correct': True, 'tuned_answer': '피이스브 플라워 페인트', 'tuned_correct': False}
{'question': '임시 취업 허가 건수는 얼마입니까?', 'gold': '553197', 'base_answer': '553,197', 'base_correct': True, 'tuned_answer': '69,177', 'tuned_correct': False}
{'question': '불만족 비율은 얼마입니까?\n', 'gold': '33', 'base_answer': '33', 'base_correct': True, 'tuned_answer': '33', 'tuned_correct': True}
{'question': '5년간 증축 항목의 합은 얼마인가요?', 'gold': '43', 'base_answer': '147', 'base_correct': False, 'tuned_answer': '30.6', 'tuned_correct': False}
{'question': '같은 값을 가진 막대는 무엇인가요?', 'gold': '덴마크, 프랑스령 폴리네시아, 오스트리아', 'base_answer': '오스트리아, 댄마크, 프랑스형 플리네시아', 'base_correct': False, 'tuned_answer': '우스 트리, 멘마, 프랑스형 플리네시아', 'tuned_correct': False}


In [5]:
base_acc = sum(r["base_correct"] for r in rows) / len(rows)
tuned_acc = sum(r["tuned_correct"] for r in rows) / len(rows)

print(f"base 모델 정확도:  {base_acc:.0%}")
print(f"QLoRA 모델 정확도: {tuned_acc:.0%}")


base 모델 정확도:  32%
QLoRA 모델 정확도: 38%


## 결과 해석
- base 모델은 정답 자체는 맞혀도 부연 설명을 붙이거나 형식(단위, 소수점 등)이 흔들리는 경우가 많습니다.
- QLoRA 모델은 학습 데이터의 응답 포맷(간결한 단답)을 그대로 재현하는 경향이 강해져 `judge_answer`(ChartQA relaxed accuracy) 기준 정확도가 올라가는 것을 확인할 수 있습니다.
- 실무에서는 이런 자동 채점 대신, 실제 라벨과의 의미적 일치 여부를 사람이 검수하거나 별도 평가 모델을 활용하는 것이 좋습니다.